# Sequence Length Comparison and Generalization

Single notebook workflow:
1. Run sequence-length benchmark and save results to disk.
2. Load results from disk.
3. Provide plots and tables comparing performance, latency, and memory usage across different sequence lengths and models.

In [ ]:
import hashlib
import json
from pathlib import Path

from IPython.display import display
from pfns.utils import get_default_device

from pfns.experiments.seq_len import (
    DEFAULT_BUCKET_BINS,
    DEFAULT_BUCKET_LABELS,
    MODEL_FAMILIES,
    add_numeric_buckets,
    build_model_style_map,
    compute_mean_rank_tables,
    download_results_bundle_from_wandb,
    evaluate_models_over_seqlens,
    get_models_from_families,
    get_models_from_names,
    load_models_for_benchmark,
    load_results_bundle,
    make_bundle_path,
    make_model_artifact_name,
    merge_model_results,
    nested_metric_table_to_long_df,
    plot_curves_from_df,
    run_metadata_matches,
    sanitize_wandb_artifact_component,
    save_results_bundle,
    upload_results_bundle_to_wandb,
)

import torch

EXPERIMENT = {
    "name": "seq_len_comparison",
    "num_classes": 5,
    "num_features": 10,
    "num_test_samples": 100,
    "num_repetitions": 100,
    "data_generation_seed": 42,
    "use_warmup_iters": False,
    "print_timing": False,
    "seqlen_list": [250, 500, 750, 1_000, 2_000, 4_000, 8_000, 16_000, 32_000, 64_000, 128_000],
    "autocast_models": {
        "DeltaNet_Cached": torch.bfloat16,
        "DeltaNet_Causal": torch.bfloat16,
        "DeltaNet_Teacher_Forcing": torch.bfloat16
    },  # Set to "auto" to infer from model configs, or specify dtypes directly here.
}

WANDB = {
    "enabled": True,
    "artifact_name": "base_results",
    "overwrite": False,
}

EXPECTED_RUN_METADATA_KEYS = (
    "seqlen_list",
    "num_features",
    "num_classes",
    "number_of_test_samples",
    "number_of_repetitions",
    "device",
    "data_generation_seed",
)

BUNDLE_ROOT = Path.cwd().resolve() / "exp_outputs" / "seq_len" / "bundles"
BUNDLE_ROOT.mkdir(parents=True, exist_ok=True)

print(f"Bundles are stored in: {BUNDLE_ROOT}")
print(f"Available model families: {list(MODEL_FAMILIES)}")

# Example by family:
models_to_compare = get_models_from_families(["transformer"])

# models_to_compare = get_models_from_names([
#     "Softmax_Transformer",
#     "KDA_causal",
#     "GLA_Cached",
#     "DeltaNet_Cached",
#     "Gated_DeltaNet_Cached_seq_len_10K",
#     "Rebased",
#     "Linear_Attention",
# ])


def _single_model_hash(model_name: str, model_config: dict) -> str:
    payload = {
        "experiment": EXPERIMENT,
        "model_name": model_name,
        "model_config": model_config,
    }
    return hashlib.sha256(
        json.dumps(payload, sort_keys=True, default=str).encode("utf-8")
    ).hexdigest()[:16]


device = get_default_device()
print(f"Using device: {device}")

expected_run_metadata = {
    "seqlen_list": list(EXPERIMENT["seqlen_list"]),
    "num_features": EXPERIMENT["num_features"],
    "num_classes": EXPERIMENT["num_classes"],
    "number_of_test_samples": EXPERIMENT["num_test_samples"],
    "number_of_repetitions": EXPERIMENT["num_repetitions"],
    "device": device,
    "data_generation_seed": EXPERIMENT["data_generation_seed"],
}


## Running the sequence-length benchmark


In [ ]:
results_by_model = {}
model_bundle_paths = {}
per_model_root = BUNDLE_ROOT / "per_model"
per_model_root.mkdir(parents=True, exist_ok=True)

if WANDB["enabled"] and WANDB["overwrite"]:
    print("WANDB overwrite=True: skipping per-model download and forcing rerun.")

for model_name, model_config in models_to_compare.items():
    model_hash = _single_model_hash(model_name, model_config)
    model_artifact_name = make_model_artifact_name(
        base_artifact_name=WANDB["artifact_name"],
        model_name=model_name,
        model_hash=model_hash,
    )

    reused_cached_result = False
    if WANDB["enabled"] and not WANDB["overwrite"]:
        cached_bundle_path = download_results_bundle_from_wandb(
            artifact_name=model_artifact_name,
            download_root=BUNDLE_ROOT / "wandb_model_cache",
        )
        if cached_bundle_path is not None:
            cached_bundle = load_results_bundle(cached_bundle_path)
            has_model = (
                model_name in cached_bundle["metric_table"]
                and model_name in cached_bundle["timing_table"]
            )
            run_metadata = cached_bundle.get("metadata", {})
            metadata_ok = run_metadata_matches(
                run_metadata,
                expected=expected_run_metadata,
                keys=EXPECTED_RUN_METADATA_KEYS,
            )

            if has_model and metadata_ok:
                results_by_model[model_name] = {
                    "schema_version": cached_bundle["bundle_metadata"].get("schema_version"),
                    "metric_table": {model_name: cached_bundle["metric_table"][model_name]},
                    "timing_table": {model_name: cached_bundle["timing_table"][model_name]},
                    "memory_table": {
                        model_name: cached_bundle["memory_table"].get(model_name, {})
                    },
                    "oom_errors": {
                        model_name: cached_bundle["oom_errors"].get(model_name, [])
                    },
                    "metadata": run_metadata,
                }
                model_bundle_paths[model_name] = cached_bundle_path
                reused_cached_result = True
                print(f"Reused cached W&B result for {model_name}: {cached_bundle_path}")
            else:
                print(
                    f"Cached artifact for {model_name} is incompatible "
                    f"(has_model={has_model}, metadata_ok={metadata_ok}). Rerunning model."
                )

    if reused_cached_result:
        continue

    print(f"Running benchmark for model: {model_name}")
    models, configs = load_models_for_benchmark({model_name: model_config}, device=device)
    model_results = evaluate_models_over_seqlens(
        models=models,
        configs=configs,
        seqlen_list=EXPERIMENT["seqlen_list"],
        num_features=EXPERIMENT["num_features"],
        num_classes=EXPERIMENT["num_classes"],
        number_of_test_samples=EXPERIMENT["num_test_samples"],
        number_of_repetitions=EXPERIMENT["num_repetitions"],
        use_warmup_iters=EXPERIMENT["use_warmup_iters"],
        print_timing=EXPERIMENT["print_timing"],
        autocast_models={model_name: EXPERIMENT["autocast_models"].get(model_name, torch.float32)},  # Use "auto" to infer from configs, or provide a dict
        device=device,
        data_generation_seed=EXPERIMENT["data_generation_seed"],
        progress_desc=f"{model_name} progress",
        forward_models= ["Non-Causal_TabPFN", "Causal_TabPFN", "Test_To_Train_Only_TabPFN"]
    )
    results_by_model[model_name] = model_results

    model_bundle_path = make_bundle_path(
        per_model_root,
        f"{EXPERIMENT['name']}_{sanitize_wandb_artifact_component(model_name)}",
    )
    save_results_bundle(
        model_results,
        model_bundle_path,
        experiment=EXPERIMENT,
        include_raw_torch=True,
    )
    model_bundle_paths[model_name] = model_bundle_path
    print(f"Saved per-model bundle for {model_name}: {model_bundle_path}")

    if WANDB["enabled"]:
        artifact_ref = upload_results_bundle_to_wandb(
            model_bundle_path,
            artifact_name=model_artifact_name,
            run_name=(
                f"{EXPERIMENT['name']}_{sanitize_wandb_artifact_component(model_name)}_"
                f"{model_hash}"
            ),
            metadata={
                "experiment": EXPERIMENT,
                "model_name": model_name,
                "model_config": model_config,
                "model_hash": model_hash,
                "run_metadata": model_results.get("metadata", {}),
            },
        )
        print(f"Uploaded per-model artifact for {model_name}: {artifact_ref}")

results = merge_model_results(
    results_by_model,
    merged_metadata={
        **expected_run_metadata,
        "models": list(models_to_compare.keys()),
    },
)

metric_df = nested_metric_table_to_long_df(results["metric_table"])
timing_df = nested_metric_table_to_long_df(results["timing_table"])
memory_df = nested_metric_table_to_long_df(results["memory_table"])

bundle_metadata = {
    "schema_version": results.get("schema_version"),
    "experiment": EXPERIMENT,
    "run_metadata": results.get("metadata", {}),
    "row_counts": {
        "metric": int(len(metric_df)),
        "timing": int(len(timing_df)),
        "memory": int(len(memory_df)),
    },
    "per_model_bundle_paths": {k: str(v) for k, v in model_bundle_paths.items()},
}


## Load benchmark results

In [ ]:
if "results" not in globals():
    raise RuntimeError("No in-memory merged results found. Run the benchmark cell first.")

print("Using in-memory merged results from per-model artifacts/runs.")

print("Bundle metadata:")
display(bundle_metadata)

print("metric_df shape:", metric_df.shape)
print("timing_df shape:", timing_df.shape)
print("memory_df shape:", memory_df.shape)

display(metric_df.head())


# Performance, latency and memory curves.

In [ ]:
PLOT_SPECS = {
    "metric": [
        ("acc", "Accuracy"),
        ("ce", "Cross Entropy Loss"),
        ("roc_auc", "ROC AUC"),
    ],
    "timing": [
        ("forward_time_ms", "Total Time (ms)"),
        ("fit_time_ms", "Fit Time (ms)"),
        ("predict_time_ms", "Predict Time (ms)"),
    ],
    "memory": [
        ("peak_allocated_mb", "Peak Allocated MB"),
        ("peak_reserved_mb", "Peak Reserved MB"),
        ("context_size_mb", "Context Size MB"),
    ],
}

PLOT_PANEL_CONFIGS = [
    ("metric", metric_df, {"title_suffix": " vs Number of In-context Samples"}),
    ("timing", timing_df, {"show_std": True, "title_suffix": " vs Number of In-context Samples"}),
    ("memory", memory_df, {"title_suffix": " vs Number of In-context Samples"}),
]

TABLE_PANEL_CONFIGS = [
    ("Performance", metric_df, {"value_fmt": "{:.4f}", "cmap": "Greens_r"}),
    ("Timing", timing_df, {}),
    ("Memory", memory_df, {}),
]

def plot_panel(df, specs, *, value_col="value", show_std=False, log_x=False, invert_y=False, title_suffix=""):
    return plot_curves_from_df(
        df,
        specs=specs,
        style_map=style_map,
        x_col="seqlen",
        value_col=value_col,
        x_label="Number of In-context Samples",
        title_suffix=title_suffix,
        show_std=show_std,
        log_x=log_x,
        invert_y=invert_y,
    )

def _display_pivot_by_metric(df, *, panel_name: str, pivot_col: str, value_col: str = "value", value_fmt: str = "{:.3f}", cmap: str = "Blues_r"):
    if df.empty:
        print(f"No {panel_name} data available.")
        return

    for metric in sorted(df["metric"].unique()):
        print(f"{panel_name} | {metric}")
        table = df[df["metric"] == metric].pivot_table(
            index="model",
            columns=pivot_col,
            values=value_col,
            observed=True,
        )
        display(table.style.format(value_fmt).background_gradient(cmap=cmap, axis=None))

def display_panel_tables(df, *, panel_name: str, value_fmt: str = "{:.3f}", cmap: str = "Blues_r"):
    if df.empty:
        print(f"No {panel_name} data available.")
        return

    print(f"=== {panel_name}: Mean by model ===")
    overall = (
        df.groupby(["model", "metric"], observed=True)["value"]
        .mean()
        .reset_index()
        .pivot_table(index="model", columns="metric", values="value", observed=True)
        .sort_index()
    )
    display(overall.style.format(value_fmt).background_gradient(cmap=cmap, axis=None))

    bucket_df = add_numeric_buckets(
        df,
        value_col="seqlen",
        bucket_col="bucket",
        bins=DEFAULT_BUCKET_BINS,
        labels=DEFAULT_BUCKET_LABELS,
    )
    print(f"=== {panel_name}: Mean by sequence-length bucket ===")
    _display_pivot_by_metric(bucket_df, panel_name=panel_name, pivot_col="bucket", value_col="value", value_fmt=value_fmt, cmap=cmap)

    print(f"=== {panel_name}: Mean by sequence length ===")
    _display_pivot_by_metric(df, panel_name=panel_name, pivot_col="seqlen", value_col="value", value_fmt=value_fmt, cmap=cmap)

style_map = build_model_style_map(sorted(metric_df["model"].unique()))

for panel_name, df, opts in PLOT_PANEL_CONFIGS:
    plot_panel(df, PLOT_SPECS[panel_name], **opts)


In [ ]:
rank_tables = compute_mean_rank_tables(metric_df, x_col="seqlen")

plot_panel(
    rank_tables["x_ranks"],
    PLOT_SPECS["metric"],
    value_col="rank",
    log_x=True,
    invert_y=True,
    title_suffix=" Rank (1 is best)",
)


# Tabels with the detailed results for each model and sequence length.

## Performance, Timing and Memory Tables

Summary tables by model, by sequence-length bucket, and by exact sequence length.


In [ ]:
for panel_name, df, kwargs in TABLE_PANEL_CONFIGS:
    display_panel_tables(df, panel_name=panel_name, **kwargs)


## Performance rank table

In [ ]:
overall_ranks = rank_tables["overall_ranks"]
bucket_ranks = rank_tables["bucket_ranks"]
x_ranks = rank_tables["x_ranks"]

for metric in sorted(metric_df["metric"].unique()):
    print(f"=== Mean rank for {metric} (lower is better) ===")
    overall = (
        overall_ranks[overall_ranks["metric"] == metric]
        .sort_values("rank")
        .drop(columns="metric")
        .reset_index(drop=True)
    )
    display(overall.style.format({"rank": "{:.2f}"}).background_gradient(subset=["rank"], cmap="Greens_r"))

print("=== Mean rank by bucket ===")
_display_pivot_by_metric(
    bucket_ranks,
    panel_name="Rank by bucket",
    pivot_col="bucket",
    value_col="rank",
    value_fmt="{:.2f}",
    cmap="Greens_r",
)

print("=== Mean rank by sequence length ===")
_display_pivot_by_metric(
    x_ranks,
    panel_name="Rank by sequence length",
    pivot_col="seqlen",
    value_col="rank",
    value_fmt="{:.2f}",
    cmap="Greens_r",
)


plot_panel(
    x_ranks,
    PLOT_SPECS["metric"],
    value_col="rank",
    log_x=True,
    invert_y=True,
    title_suffix=" Rank (1 is best)",
)
